In [19]:
import sys
if 'google.colab' in sys.modules:
    from IPython.core.getipython import get_ipython
    get_ipython().run_line_magic("pip", "install transformers sentencepiece accelerate")
    get_ipython().run_line_magic("pip", "install git+https://github.com/UlisseMini/activation_additions_hf")

In [31]:
import torch
import activation_additions as aa

from typing import Dict, Union, Callable
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector
from functools import lru_cache

In [32]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

C:\Users\itayb\AppData\Local\Temp\ipykernel_42656\1209224586.py:1: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"


'cuda'

In [33]:
MODEL = "openai-community/gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 1.0,
    "top_p": 0.3,
    "freq_penalty": 1.0,
    "num_comparisons": 1,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

In [35]:
@lru_cache(maxsize=1000)
def get_diff_vector(prompt_add: str, prompt_sub: str, layer: int):
    return aa.get_diff_vector(model, tokenizer, prompt_add, prompt_sub, layer)


@lru_cache
def run_aa(coeff: float, layer: int, prompt_add: str, prompt_sub: str, texts: tuple[str], loss_ignore_mod_tokens: bool = False):
    # todo: could compute act_diff for all layers at once, then a single fwd pass of cost for changing layer.
    act_diff = coeff * get_diff_vector(prompt_add, prompt_sub, layer)
    blocks = aa.get_blocks(model)
    with aa.pre_hooks([(blocks[layer], aa.get_hook_fn(act_diff))]):
        inputs = tokenizer(list(texts), return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        output = model(**inputs,output_hidden_states=True, return_dict=True)

    logprobs = torch.log_softmax(output.logits.to(torch.float32), -1)
    token_loss = -logprobs[..., :-1, :].gather(dim=-1, index=inputs['input_ids'][..., 1:, None])[..., 0]
    return token_loss

In [36]:
def calculate_perplexity_change(args_obj):
    assert len(args_obj['texts'][0]) == len(args_obj['texts'][1]), 'must have same number of positive/negative examples'
    split_at = len(args_obj['texts'][0])
    args_obj['texts'] = args_obj['texts'][0] + args_obj['texts'][1]

    token_loss = run_aa(args_obj["coeff"],args_obj["layer"],args_obj["prompt_add"],args_obj["prompt_sub"],args_obj["texts"],args_obj["loss_ignore_mod_tokens"])
    abs_token_loss = run_aa(0., 0, ' ', ' ', texts=args_obj['texts']) # cached
    perplexity_steered = torch.exp(1/2*(token_loss[:split_at].mean()))
    perplexity_unsteered = torch.exp(1/2*(abs_token_loss[:split_at].mean()))
    return (perplexity_unsteered-perplexity_steered).item()

In [37]:
summand_dict = {
    "coeff":2.5,
    "layer":6,
    "prompt_add":"love",
    "prompt_sub":"hate",
    "texts": (('I hate you because I love you\nI love you', "I hate you because you're so beautiful.", 'I hate you because I love you\nThe world is a stage'), ('I hate you because you are a girl.', "I hate you because you're not me.\nI hate you because I am me.", 'I hate you because you are a man and I am a woman.')),
    "loss_ignore_mod_tokens":False
}
calculate_perplexity_change(summand_dict)

-2.345071792602539